In [1]:
import requests
import pandas as pd
from typing import Iterable, Optional

In [2]:
URL = "https://www.cbioportal.org/api"


In [3]:
def get_mutations_from_samples(
    sample_ids: Iterable[str],
    study_id: str,
    molecular_profile_id: Optional[str] = None,
    session: Optional[requests.Session] = None,
    timeout: int = 60,
) -> pd.DataFrame:
    """
    Fetch mutation records from cBioPortal for a list of sample IDs.

    Parameters
    ----------
    sample_ids : Iterable[str]
        cBioPortal sample IDs, e.g. ["TCGA-GC-A3BM-01", "TCGA-XF-A9SY-01"].
    study_id : str
        cBioPortal study ID, e.g. "blca_tcga".
    molecular_profile_id : str | None
        Mutation profile ID. If None, defaults to "{study_id}_mutations".
    base_url : str
        cBioPortal API base URL.
    session : requests.Session | None
        Optional requests session.
    timeout : int
        Request timeout in seconds.

    Returns
    -------
    pd.DataFrame
        Mutation table. Empty DataFrame if nothing is returned.

    Notes
    -----
    - This function assumes all sample_ids belong to the same study.
    - If your samples come from multiple studies, call the function per study.
    """

    sample_ids = [str(x).strip() for x in sample_ids if str(x).strip()]
    if not sample_ids:
        raise ValueError("sample_ids is empty.")

    if molecular_profile_id is None:
        molecular_profile_id = f"{study_id}_mutations"

    http = session or requests.Session()

    url = f"{URL}/molecular-profiles/{molecular_profile_id}/mutations/fetch"
    print(">>>", url)

    payload = {
        "sampleIds": sample_ids
    }

    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

    resp = http.post(url, json=payload, headers=headers, timeout=timeout)

    # Helpful error message from cBioPortal
    if not resp.ok:
        msg = ""
        try:
            msg = resp.json()
        except Exception:
            msg = resp.text
        raise RuntimeError(
            f"cBioPortal request failed: HTTP {resp.status_code} | "
            f"profile={molecular_profile_id} | details={msg}"
        )

    data = resp.json()
    if not data:
        return pd.DataFrame()

    df = pd.DataFrame(data)

    # Optional: reorder a few common columns if present
    preferred = [
        "sampleId",
        "patientId",
        "studyId",
        "molecularProfileId",
        "gene",
        "entrezGeneId",
        "proteinChange",
        "mutationType",
        "mutationStatus",
        "variantType",
        "chr",
        "startPosition",
        "endPosition",
        "referenceAllele",
        "tumorSeqAllele2",
    ]
    cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
    return df[cols]

In [4]:
sample_ids = [
    "TCGA-GC-A3BM-01",
    "TCGA-XF-A9SY-01",
]

df_mut = get_mutations_from_samples(sample_ids=sample_ids, study_id="blca_tcga")
len(df_mut)

>>> https://www.cbioportal.org/api/molecular-profiles/blca_tcga_mutations/mutations/fetch


103

In [5]:
df_mut.columns

Index(['sampleId', 'patientId', 'studyId', 'molecularProfileId',
       'entrezGeneId', 'proteinChange', 'mutationType', 'mutationStatus',
       'variantType', 'chr', 'startPosition', 'endPosition', 'referenceAllele',
       'uniqueSampleKey', 'uniquePatientKey', 'center', 'validationStatus',
       'tumorAltCount', 'tumorRefCount', 'ncbiBuild', 'keyword',
       'variantAllele', 'refseqMrnaId', 'proteinPosStart', 'proteinPosEnd'],
      dtype='object')

In [6]:
df_mut

,sampleId,patientId,studyId,molecularProfileId,entrezGeneId,proteinChange,mutationType,mutationStatus,variantType,chr,...,center,validationStatus,tumorAltCount,tumorRefCount,ncbiBuild,keyword,variantAllele,refseqMrnaId,proteinPosStart,proteinPosEnd
0,TCGA-GC-A3BM-01,TCGA-GC-A3BM,blca_tcga,blca_tcga_mutations,148,F385L,Missense_Mutation,Somatic,SNP,8,...,broad.mit.edu,Untested,106,118,GRCh37,ADRA1A F385 missense,T,NM_033303.3,385,385
1,TCGA-GC-A3BM-01,TCGA-GC-A3BM,blca_tcga,blca_tcga_mutations,773,R935W,Missense_Mutation,Somatic,SNP,19,...,broad.mit.edu,Untested,16,20,GRCh37,CACNA1A R935 missense,A,NM_001127222.1,935,935
2,TCGA-GC-A3BM-01,TCGA-GC-A3BM,blca_tcga,blca_tcga_mutations,996,I91Sfs*54,Frame_Shift_Del,Somatic,DEL,17,...,broad.mit.edu,Untested,9,154,GRCh37,CDC27 truncating,-,NA,91,91
3,TCGA-GC-A3BM-01,TCGA-GC-A3BM,blca_tcga,blca_tcga_mutations,996,S93A,Missense_Mutation,Somatic,SNP,17,...,broad.mit.edu,Untested,8,142,GRCh37,CDC27 S93 missense,C,NA,93,93
4,TCGA-GC-A3BM-01,TCGA-GC-A3BM,blca_tcga,blca_tcga_mutations,1029,G89Lfs*30,Frame_Shift_Del,Somatic,DEL,9,...,broad.mit.edu,Valid,17,2,GRCh37,CDKN2A truncating,-,NM_000077.4,89,89
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,TCGA-GC-A3BM-01,TCGA-GC-A3BM,blca_tcga,blca_tcga_mutations,283208,L230F,Missense_Mutation,Somatic,SNP,11,...,broad.mit.edu,Untested,48,87,GRCh37,P4HA3 L230 missense,G,NM_182904.3,230,230
99,TCGA-GC-A3BM-01,TCGA-GC-A3BM,blca_tcga,blca_tcga_mutations,283554,N418Ifs*7,Frame_Shift_Del,Somatic,DEL,14,...,broad.mit.edu,Untested,15,52,GRCh37,GPR137C truncating,-,NM_001099652.1,418,418
100,TCGA-GC-A3BM-01,TCGA-GC-A3BM,blca_tcga,blca_tcga_mutations,350383,S270L,Missense_Mutation,Somatic,SNP,17,...,broad.mit.edu,Untested,23,22,GRCh37,GPR142 S270 missense,T,NM_181790.1,270,270
101,TCGA-GC-A3BM-01,TCGA-GC-A3BM,blca_tcga,blca_tcga_mutations,375248,N59S,Missense_Mutation,Somatic,SNP,2,...,broad.mit.edu,Untested,30,27,GRCh37,ANKRD36 N59 missense,G,NM_001164315.1,59,59


In [ ]:
sample_ids = ['TCGA-A1-A0SB-01A',
 'TCGA-A1-A0SB-01Z',
 'TCGA-A7-A26E-01A',
 'TCGA-A7-A26E-01B',
 'TCGA-A7-A26E-01Z',
 'TCGA-A8-A07W-01A',
 'TCGA-A8-A07W-01Z',
 'TCGA-AN-A0AM-01A',
 'TCGA-AN-A0AM-01Z',
 'TCGA-E2-A1IU-01A',
 'TCGA-E2-A1IU-01Z']

df_mut = get_mutations_from_samples(sample_ids=sample_ids, study_id="brca_tcga")
len(df_mut)